# Table Generation

This notebook builds a summary table that averages model-comparison results across splits and combines them with BFS and optimal-move averages.

In [9]:
from pathlib import Path
import functools
import re

import pandas as pd
import matplotlib.pyplot as plt


TABLE_FILENAME = "table_val_bfs_opt_sim_22_2_120_top1_mtfnone_hard_v_hard.png"
MODEL_COMPARISON_ROOT = Path(r"c:\Users\Michael Migacev\Desktop\UNI\Wirklich_UNI\AICON\master-thesis-tol\master-thesis-tol\tower_of_london_plots\problem_ordering_outputs\po_outputs_22_2_mtfnone_extra_number_of_moves")
MODEL_COMPARISON_ROWS = 1
BFS_ROOT = Path(r"c:\Users\Michael Migacev\Desktop\UNI\Wirklich_UNI\AICON\master-thesis-tol\master-thesis-tol\tower_of_london_plots\problem_ordering_outputs\po_metric_bfs_22_2_120")
BFS_SIMPLE_ROOT = Path(r"c:\Users\Michael Migacev\Desktop\UNI\Wirklich_UNI\AICON\master-thesis-tol\master-thesis-tol\tower_of_london_plots\problem_ordering_outputs\po_metric_bfs_simple_22_2_120")
OPT_MOV_ROOT = Path(r"c:\Users\Michael Migacev\Desktop\UNI\Wirklich_UNI\AICON\master-thesis-tol\master-thesis-tol\tower_of_london_plots\problem_ordering_outputs\po_metric_opt_mov_22_2_120")
RUN_SCRIPT = True

COMPARISON_METRICS = ["success_rate_rev", "mean_extra_moves_on_success"]
PATIENT_GROUPS = ["Healthy", "MCI", "PD", "Stroke"]
SPLIT_PATTERN = re.compile(r"^split_\d+$")
SPLIT_SELECTION = "hard_v_hard"  # all | easy_v_easy | hard_v_hard | easy_v_hard
EASY_PROBLEMS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 16]
HARD_PROBLEMS = [10, 11, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24]
MODEL_TRAIN_CANDIDATES = ("top*_training_extended*.csv", "top25_training.csv", "top_25_training.csv", "top5_training.csv")
MODEL_VALIDATION_CANDIDATES = ("top*_validation_extended*.csv", "top25_validation.csv", "top_25_validation.csv", "top5_validation.csv")


def sorted_split_dirs(root: Path) -> list[Path]:
    if not root.exists():
        raise FileNotFoundError(f"Folder does not exist: {root}")

    split_dirs = [path for path in root.iterdir() if path.is_dir() and SPLIT_PATTERN.match(path.name)]
    return sorted(split_dirs, key=lambda path: int(path.name.split("_")[-1]))


def _extract_problem_numbers(value: str) -> set[int]:
    return {int(number) for number in re.findall(r"\d+", str(value))}


def _find_validation_file(split_dir: Path) -> Path:
    candidate_files = sorted(split_dir.rglob("top*_validation_extended*.csv"))
    if not candidate_files:
        raise FileNotFoundError(f"No validation CSV found in split folder: {split_dir}")
    return candidate_files[0]


def _load_validation_problem_ids(validation_csv_path: Path) -> set[int]:
    column_frame = pd.read_csv(validation_csv_path, nrows=0)
    validation_ids = set()
    for column_name in column_frame.columns:
        column_text = str(column_name).strip()
        if column_text.startswith("TOL-ID-"):
            validation_ids.update(_extract_problem_numbers(column_text))
    if not validation_ids:
        raise ValueError(f"No validation problem columns were found in {validation_csv_path}")
    invalid_validation = [number for number in sorted(validation_ids) if number < 1 or number > 24]
    if invalid_validation:
        raise ValueError(
            "Validation CSV contains values outside [1, 24]: "
            + ", ".join(str(value) for value in invalid_validation)
        )
    return validation_ids


def _split_matches_selection(validation_ids: set[int]) -> bool:
    easy_set = set(EASY_PROBLEMS)
    hard_set = set(HARD_PROBLEMS)

    if SPLIT_SELECTION == "all":
        return True
    if SPLIT_SELECTION == "easy_v_easy":
        return validation_ids.issubset(easy_set)
    if SPLIT_SELECTION == "hard_v_hard":
        return validation_ids.issubset(hard_set)
    if SPLIT_SELECTION == "easy_v_hard":
        return bool(validation_ids & easy_set) and bool(validation_ids & hard_set)
    raise ValueError("SPLIT_SELECTION must be one of: all, easy_v_easy, hard_v_hard, easy_v_hard")


@functools.lru_cache(maxsize=1)
def _selected_split_names() -> tuple[str, ...]:
    selected_names: list[str] = []
    for split_dir in sorted_split_dirs(MODEL_COMPARISON_ROOT):
        validation_csv_path = _find_validation_file(split_dir)
        validation_ids = _load_validation_problem_ids(validation_csv_path)
        if _split_matches_selection(validation_ids):
            selected_names.append(split_dir.name)

    if not selected_names:
        raise ValueError(f"No splits matched SPLIT_SELECTION={SPLIT_SELECTION!r}")

    return tuple(selected_names)


def selected_split_count() -> int:
    return len(_selected_split_names())


def select_split_dirs(root: Path) -> list[Path]:
    selected_names = set(_selected_split_names())
    split_dirs = [path for path in sorted_split_dirs(root) if path.name in selected_names]
    if not split_dirs:
        raise ValueError(f"No selected splits found in {root}")
    return split_dirs


def find_first_existing_file(folder: Path, candidates: tuple[str, ...]) -> Path:
    for candidate in candidates:
        if "*" in candidate:
            glob_results = sorted(folder.glob(candidate))
            if glob_results:
                return glob_results[0]
        else:
            candidate_path = folder / candidate
            if candidate_path.exists():
                return candidate_path
    raise FileNotFoundError(f"None of {candidates} found in {folder}")


def read_top_rows_average(csv_path: Path, top_rows: int) -> dict[str, float]:
    if top_rows < 1:
        raise ValueError("MODEL_COMPARISON_ROWS must be at least 1")

    data_frame = pd.read_csv(csv_path)
    required_columns = {"mean", "std"}
    missing_columns = required_columns - set(data_frame.columns)
    if missing_columns:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing_columns)}")

    selected_rows = data_frame.head(top_rows)
    return {
        "mean": pd.to_numeric(selected_rows["mean"], errors="coerce").mean(),
        "std": pd.to_numeric(selected_rows["std"], errors="coerce").mean(),
    }


def _collect_top_row_means(csv_path: Path, top_rows: int) -> list[float]:
    if top_rows < 1:
        raise ValueError("MODEL_COMPARISON_ROWS must be at least 1")

    data_frame = pd.read_csv(csv_path)
    if "mean" not in data_frame.columns:
        raise ValueError(f"{csv_path} is missing column: mean")

    selected_rows = data_frame.head(top_rows)
    mean_series = pd.to_numeric(selected_rows["mean"], errors="coerce").dropna()
    return mean_series.tolist()


def collect_model_comparison_row(root: Path, comparison_metric: str, patient_group: str, top_rows: int) -> dict[str, float | str]:
    train_values = []
    test_values = []
    for split_dir in select_split_dirs(root):
        result_dir = split_dir / f"{comparison_metric}_{patient_group}"
        if not result_dir.exists():
            raise FileNotFoundError(f"Missing model-comparison folder: {result_dir}")

        training_file = find_first_existing_file(result_dir, MODEL_TRAIN_CANDIDATES)
        validation_file = find_first_existing_file(result_dir, MODEL_VALIDATION_CANDIDATES)

        train_values.extend(_collect_top_row_means(training_file, top_rows))
        test_values.extend(_collect_top_row_means(validation_file, top_rows))

    train_series = pd.Series(train_values, dtype="float64")
    test_series = pd.Series(test_values, dtype="float64")
    if train_series.empty or test_series.empty:
        raise ValueError(f"No model-comparison values found for {comparison_metric} / {patient_group}")

    return {
        "comparison_metric": comparison_metric,
        "patient_group": patient_group,
        "avg_train_mean": train_series.mean(),
        "avg_train_std": train_series.std(),
        "avg_test_mean": test_series.mean(),
        "avg_test_std": test_series.std(),
    }


def collect_metric_group_summary(root: Path, comparison_metric: str, patient_group: str, value_name: str) -> dict[str, float]:
    split_values = []
    file_name = f"{comparison_metric}_{patient_group}_tau_results.csv"

    for split_dir in select_split_dirs(root):
        result_file = split_dir / file_name
        if not result_file.exists():
            raise FileNotFoundError(f"Missing tau results CSV: {result_file}")

        result_frame = pd.read_csv(result_file)
        if result_frame.shape[1] < 3:
            raise ValueError(f"Expected at least 3 columns in {result_file}")

        split_values.append(
            {
                "mean": pd.to_numeric(result_frame.iloc[:, 1], errors="coerce").mean(),
                "std": pd.to_numeric(result_frame.iloc[:, 2], errors="coerce").mean(),
            }
        )

    split_frame = pd.DataFrame(split_values)
    return {
        f"avg_{value_name}_mean": split_frame["mean"].mean(),
        f"avg_{value_name}_std": split_frame["mean"].std(),
    }


def find_tables_dir(reference_path: Path) -> Path:
    for candidate_parent in [reference_path, *reference_path.parents]:
        if candidate_parent.name == "tower_of_london_plots":
            return candidate_parent / "tables"
    return reference_path.parent / "tables"


def build_summary_table() -> pd.DataFrame:
    rows = []
    for comparison_metric in COMPARISON_METRICS:
        for patient_group in PATIENT_GROUPS:
            model_summary = collect_model_comparison_row(
                MODEL_COMPARISON_ROOT,
                comparison_metric,
                patient_group,
                MODEL_COMPARISON_ROWS,
            )
            bfs_summary = collect_metric_group_summary(BFS_ROOT, comparison_metric, patient_group, "bfs")
            bfs_simple_summary = collect_metric_group_summary(BFS_SIMPLE_ROOT, comparison_metric, patient_group, "bfs_simple")
            opt_summary = collect_metric_group_summary(OPT_MOV_ROOT, comparison_metric, patient_group, "opt_mov")
            rows.append(
                {
                    "comparison_metric": model_summary["comparison_metric"],
                    "patient_group": model_summary["patient_group"],
                    "avg_train_mean": model_summary["avg_train_mean"],
                    "avg_train_std": model_summary["avg_train_std"],
                    "avg_test_mean": model_summary["avg_test_mean"],
                    "avg_test_std": model_summary["avg_test_std"],
                    "avg_bfs_mean": bfs_summary["avg_bfs_mean"],
                    "avg_bfs_std": bfs_summary["avg_bfs_std"],
                    "avg_bfs_simple_mean": bfs_simple_summary["avg_bfs_simple_mean"],
                    "avg_bfs_simple_std": bfs_simple_summary["avg_bfs_simple_std"],
                    "avg_opt_mov_mean": opt_summary["avg_opt_mov_mean"],
                    "avg_opt_mov_std": opt_summary["avg_opt_mov_std"],
                }
            )

    table_frame = pd.DataFrame(rows)
    expected_row_count = len(COMPARISON_METRICS) * len(PATIENT_GROUPS)
    if len(table_frame) != expected_row_count:
        raise ValueError(f"Expected {expected_row_count} rows, got {len(table_frame)}")

    return table_frame


def save_table_outputs(table_frame: pd.DataFrame) -> tuple[Path, Path]:
    output_dir = find_tables_dir(MODEL_COMPARISON_ROOT)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_png_path = output_dir / TABLE_FILENAME
    output_csv_path = output_dir / f"{Path(TABLE_FILENAME).stem}.csv"

    csv_frame = table_frame.copy()
    csv_frame.to_csv(output_csv_path, index=False, float_format="%.6f")

    render_frame = table_frame.round(3).copy().fillna("")
    fig_width = max(12, len(render_frame.columns) * 1.5)
    fig_height = max(4, 0.45 * len(render_frame) + 1.8)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    cell_text = []
    for _, row in render_frame.iterrows():
        formatted_row = []
        for value in row:
            if isinstance(value, float):
                formatted_row.append(f"{value:.3f}")
            else:
                formatted_row.append(str(value))
        cell_text.append(formatted_row)

    table = ax.table(
        cellText=cell_text,
        colLabels=render_frame.columns.tolist(),
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.25)

    plt.savefig(output_png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return output_png_path, output_csv_path


if RUN_SCRIPT:
    summary_table = build_summary_table()
    processed_splits = selected_split_count()
    png_path, csv_path = save_table_outputs(summary_table)
    print(summary_table)
    print(f"Processed splits: {processed_splits}")
    print(f"Saved table PNG to: {png_path}")
    print(f"Saved table CSV to: {csv_path}")

             comparison_metric patient_group  avg_train_mean  avg_train_std  \
0             success_rate_rev       Healthy        0.535882       0.028533   
1             success_rate_rev           MCI        0.637055       0.021387   
2             success_rate_rev            PD        0.630517       0.022980   
3             success_rate_rev        Stroke        0.585975       0.026021   
4  mean_extra_moves_on_success       Healthy        0.580547       0.026555   
5  mean_extra_moves_on_success           MCI        0.543030       0.023504   
6  mean_extra_moves_on_success            PD        0.627066       0.027476   
7  mean_extra_moves_on_success        Stroke        0.620332       0.041025   

   avg_test_mean  avg_test_std  avg_bfs_mean  avg_bfs_std  \
0      -0.066667      1.014833      0.296667     0.879060   
1       0.466667      0.899553      0.296667     0.879060   
2       0.400000      0.932183      0.290000     0.881359   
3       0.133333      1.008014     -0.110000